Requirement:
- Gemini API key, which you can get for free from Google AI studio. See [Online Guide](https://www.stephenwthomas.com/azure-integration-thoughts/how-to-get-free-google-gemini-api-access-step-by-step-guide-for-2025/) for help.

- Press the 🗝 Secrets tab on the left sidebar, add the API key there with name: GEMINI_API_KEY and paste the API key into the value field. Make Sure that Notebook access is enabled.

To Use:
- Press ▶ Run all at the top menu
- It will take a couple minutes to set things up (1-2 minutes)
- Scroll all the way to the bottom

NOTICE: I've read online that some people got banned from Colab for using Selenium on it, USE AT YOUR OWN RISK

In [ ]:
%%capture
# Install Python packages
!pip install google-colab-selenium

In [ ]:
import google_colab_selenium as gs
from google.genai import Client
from selenium import webdriver
from google.colab import userdata
from IPython.display import HTML
from selenium.webdriver.support.ui import WebDriverWait
from typing import List

def convert_month_to_number(month: str) -> str:
  month_dict = {
    "january": "1",
    "february": "2",
    "march": "3",
    "april": "4",
    "may": "5",
    "june": "6",
    "july": "7",
    "august": '8',
    "september": '9',
    "october": '10',
    "november": '11',
    "december": '12'}
  # TODO: this will fail if the month is not exactly as above
  return month_dict[month]

def open_url(url: str, driver: webdriver.Chrome) -> None:
  driver.get(url)
  WebDriverWait(driver, 10)
  driver.implicitly_wait(2)

def call_gemini(client: Client, prompt: str, context: str) -> str:
  response = client.models.generate_content(
      model="gemini-2.5-flash-lite",
      contents= "Execute the following command: " + prompt +
      "\n" +
      "Given the following context:" + context)

  return response.text

def validate_gemini_res(res: str, expected: List[str], topic:str) -> str:
  if res in expected:
    return res
  else:
    print("Gemini Failed to pick from standardize list for "+ topic)
    return ""

In [ ]:
from typing import List
from selenium.webdriver.common.by import By
from os import getenv
from google import genai
from google.colab import data_table
import pandas as pd
import re


# TODO:
# 1. Error checking
# Specifically:
# - Url is from events @ brown
# - Handle Selenium errors with not finding elements
# - Handle Gemini errors
# - Handle if given month other than dictonary

# Setup Gemini
api_key = userdata.get('GEMINI_API_KEY')
gemini_client = genai.Client(api_key=api_key)


def extract_event_details(url: str) -> dict:
  print("Opening URL...")
  open_url(url, driver)

  # Collect event details
  print("Extracting event details...")
  event_details = {}

  # Title is retrieved from the tab name
  event_details["title"] = driver.title.split('|')[0].strip()

  # Date is captured from the web page then converted to "MM/DD/YYYY" format
  numeric_date_list: List[str] = driver.find_element(By.ID,
                                                     "lw_cal_this_day").text.split(
    " ")
  month_num: str = convert_month_to_number(numeric_date_list[0].lower())
  day_num: str = numeric_date_list[1][:-1]
  year_num: str = numeric_date_list[2]
  event_details["date"] = month_num + "/" + day_num + "/" + year_num

  # Semester is calculated from the month
  month_num_int: int = int(month_num)

  if 8 <= month_num_int <= 12 or month_num_int == 1:
    event_details["semester"] = "Fall"
  elif 2 <= month_num_int <= 5:
    event_details["semester"] = "Spring"
  elif 6 <= month_num_int <= 8:
    event_details["semester"] = "Summer"
  else:
    raise ValueError("Invalid month num")

  # Year is inputted directly
  event_details["year"] = year_num

  # Description is extracted from the web page
  event_details["description"] = driver.find_element(By.CLASS_NAME,
                                                     "lw_calendar_event_description").text

  # Keywords are extracted by gemini from description
  event_details["keywords"] = call_gemini(gemini_client,
                                          "Extract keywords from the event description\n"
                                          "Only return the keywords separated by commas\n",
                                          event_details["description"])

  # Topics are extracted by gemini from the description
  event_details["topics"] = call_gemini(gemini_client,
                                        "Extract key themes from the event description\n"
                                        "Only return the one to three word themes separated by commas\n",
                                        event_details["description"])

  # Region is extracted by Gemini from the description
  possible_regions = ["Africa",
                      "Brazil",
                      "China",
                      "Europe",
                      "India & South Asia",
                      "Latin America & Caribbean",
                      "Middle East",
                      "Russia",
                      "United States"]
  region_res = call_gemini(gemini_client,
                                        "Extract the region of the event\n"
                                        "Only return the region name\n"
                                        "Only pick from the following\n"
                                        "Return the exact Value\n"
                                        + str(possible_regions),
                                        event_details["description"])
    ## Validate Gemini's results
  event_details["region"] = validate_gemini_res(region_res, possible_regions, "region")

  # Country is extracted by gemini from the description
  event_details["country"] = call_gemini(gemini_client,
                                        "Extract the country of the event\n"
                                        "Only return the country name\n",
                                        event_details["description"])

  # Event series is chosen from standardized list
  # TODO: Check if other is good here
  possible_series: List[str] = ["Watson Distinguished Speaker Series",
                                "Watson Institute Research Seminar Series",
                                "War in Ukraine",
                                "Security Studies Seminar",
                                "Senior Fellows",
                                "Israel-Palestine Lecture Series",
                                "OP Jindal Distinguished Lectures",
                                "Book Adda",
                                "South Asia Seminar",
                                "Art History from the South",
                                "Other"]

  event_res = call_gemini(gemini_client,
                                        "Check to see if this event is part of the following series\n"
                                        + str(possible_series) +
                                        "Only return the series name"
                                        "or Other if it doesn't match any of the series",
                                        event_details["description"])

      ## Validate Gemini's results
  event_details["event_series"] = validate_gemini_res(event_res, possible_series, "series")

  # -----------------
  # Center / Initiative / Program is chosen from standardized list
  # TODO: get the list and implement this

  # Research Theme is chosen by gemini from the description
  possible_themes = ["Research", "Security", "Development", "Governance"]
  theme_res = call_gemini(gemini_client,
                                                "Choose from: Security, Development, Governance (definitions provided below)"
                                                "Only Return the chosen theme name"
                                                "Research Theme Definitions"
                                                "Security: Covers traditional and emerging global security concerns, including climate change, pandemics, cyber threats, and post-conflict reconstruction."
                                                "Development: Focuses on inequality, governance, urban transformation, democracy, and global economic systems."
                                                "Governance: Explores how globalization affects political and economic institutions and the need for new forms of global governance.",
                                                event_details["description"])
    ## Validate Gemini Results
  event_details["research_theme"] = validate_gemini_res(theme_res, possible_themes, "research_theme")

  # The right column is retrieved from the web page
  # This column includes data for location, room, sponsor
  right_col: List[str] = driver.find_element(By.ID,
                                             "lw_cal_event_detail_cols_right").text.split(
    '\n')

  for line in right_col:
    if "Location" in line:
      event_details["Building"] = line.split(':')[1]
    elif "Room" in line:
      event_details["Location"] = line.split(':')[1]
    elif "Sponsor" in line:
      # Sponsor is the same as center
      event_details["sponsor"] = line.split(':')[1]


  # Student Run
  # NOTE: 90% of the time is no, but leave it empty for now
  event_details["student_run"] = ""

  # Privacy
  # TODO: This is dependent on whether the Youtube link is available


  # Link is inputted directly
  event_details["link"] = url

  # Watson Faculty
  # TODO: This is the description of the yt video

  # Youtube Link is extracted by gemini from the description
  # TODO: handle case when there is no youtube link
  # TODO: handle the privacy attribute from the result of the yt link
  # Right now it can't find it if the order of html elements changes
  # and fails if not found
  # event_details["yt_link"] = driver.find_element(By.XPATH, "/html/body/div[1]/main/div/div[1]/section/div[2]/div[2]/div/div/div/div[1]/a").get_attribute("href")

  # Talent is extracted by gemini from the description
  talent_list: List[str] = call_gemini(gemini_client,
                                       "Extract the talent of the event"
                                       "Only names of people attending or speaking at the event"
                                       "return separated by commas",
                                       event_details["description"]).split(",")

  for i in range(len(talent_list)):
    if i > 3:
      # Can only enter 4 talents in the spreadsheet
      print("Too many talents, only entering the first 4")
      break
    event_details["talent" + str(i + 1)] = talent_list[i]
  return event_details


if __name__ == "__main__":
  driver = gs.Chrome()

  while True:
    url = input("Enter the URL of the event from events@brown: ")
    if url.lower() == "quit" or url.lower() == "exit":
      print("Exiting....")
      break
    expected_pattern = r'^https:\/\/events\.brown\.edu\/event\/[a-zA-Z0-9\-]+$'
    if not re.match(expected_pattern, url):
      continue
    events_details = extract_event_details(url)
    df = pd.DataFrame(events_details, index=[0])

    # Display as tab-separated for pasting into Sheets
    data_table.enable_dataframe_formatter()
    tsv_output = df.to_csv(sep='\t', index=False)
    display(HTML(f"<textarea style='width:100%; height:100px;'>{tsv_output}</textarea>"))
  driver.quit()

    # Example urls for testing
    # "https://events.brown.edu/event/303266-syria-after-assad-a-teach-in"
    # "https://events.brown.edu/event/immigrationjournalism"
    # "https://events.brown.edu/event/321735-understanding-the-government-shutdown-causes-and"
    # "https://events.brown.edu/event/313473-thea-riofrancos-extraction-the-frontiers-of-green-cap"


Wait for above cell 🔼 to be ready. You should see an input box soon.
Type "quit" in it to terminate the program